In [ ]:
# Example 1: basic promise creation and resolution
from vine import promise

results = []

def on_success(value):
    results.append(('success', value))

p = promise(on_success)
p(42)

print(f"ready: {p.ready}")
print(f"failed: {p.failed}")
print(f"results: {results}")
assert p.ready is True
assert p.failed is False
assert results == [('success', 42)]

In [ ]:
# Example 2: promise chaining via then()
from vine import promise

captured = []

def step_one(x):
    captured.append(('step_one', x))
    return x * 2

def step_two(x):
    captured.append(('step_two', x))

p = promise(step_one)
p.then(promise(step_two))
p(5)

print(f"captured: {captured}")
assert ('step_one', 5) in captured
assert p.ready

In [ ]:
# Example 3: error callback via on_error
from vine import promise

errors = []

def on_error(exc):
    errors.append(type(exc).__name__)

def will_raise():
    raise ValueError('boom')

p = promise(will_raise, on_error=on_error)
try:
    p()
except ValueError:
    pass

print(f"failed: {p.failed}")
print(f"errors: {errors}")
assert p.failed is True
assert 'ValueError' in errors

In [ ]:
# Example 4: barrier — wait for multiple promises
from vine import promise, barrier

fired = []

def on_all_done():
    fired.append('done')

p1 = promise()
p2 = promise()
p3 = promise()

b = barrier([p1, p2, p3], callback=promise(on_all_done))
p1()
p2()
assert fired == []
p3()

print(f"fired: {fired}")
assert fired == ['done']

In [ ]:
# Example 5: wrap a callable with @promise via maybe_promise + funs
from vine import promise, maybe_promise, ppartial, transform

# maybe_promise: pass-through if already a promise, else wrap
p = maybe_promise(lambda x: x + 1)
assert isinstance(p, promise)

# ppartial: partial application that returns a promise
captured = []
p2 = ppartial(lambda a, b: captured.append(a + b), 10)
p2(5)
assert captured == [15]

# transform: apply a transformation callable in front of a promise
out = []
p3 = transform(lambda v: v.upper(), promise(lambda v: out.append(v)))
p3('hello')
print(f"transformed output: {out}")
assert out == ['HELLO']